In [1]:
%pip install duckdb pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Combine cell instances and normalize columns for analysis

import duckdb

data_dir = "data/gtex-private"
query = f"""
SELECT
  split_part(filename, '/', 3) as dataset,
  Organ_ID as organ,
  split_part(filename, '/', 4) as tool,
  column00 as cell,
  clid as cell_id,
  CL_Label as cell_label,
  match_type,
  COALESCE("mapping.score", conf_score, popv_prediction_score / 6, 0) as confidence_score,
FROM read_csv('{ data_dir }/*/*/annotations.csv', union_by_name = true, filename = true)
"""

cells = duckdb.sql(query)
cells.write_csv(f'{ data_dir }/cell-instances.csv.gz')
cells.show()

┌────────────────────┬────────────────┬─────────┬──────────────────────────┬────────────┬────────────────────────┬─────────────────┬────────────────────┐
│      dataset       │     organ      │  tool   │           cell           │  cell_id   │       cell_label       │   match_type    │  confidence_score  │
│      varchar       │    varchar     │ varchar │         varchar          │  varchar   │        varchar         │     varchar     │       double       │
├────────────────────┼────────────────┼─────────┼──────────────────────────┼────────────┼────────────────────────┼─────────────────┼────────────────────┤
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGCACGTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch │ 0.7471968103744316 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGGACCTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch │ 0.8620346211761541 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGGCGATA-1-1JKYN │

In [3]:
# Show only cells annotated with azimuth
cells.filter("tool = 'azimuth'")

┌────────────────────┬────────────────┬─────────┬──────────────────────────┬────────────┬────────────────────────┬─────────────────────┬────────────────────┐
│      dataset       │     organ      │  tool   │           cell           │  cell_id   │       cell_label       │     match_type      │  confidence_score  │
│      varchar       │    varchar     │ varchar │         varchar          │  varchar   │        varchar         │       varchar       │       double       │
├────────────────────┼────────────────┼─────────┼──────────────────────────┼────────────┼────────────────────────┼─────────────────────┼────────────────────┤
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGCACGTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch     │ 0.7471968103744316 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGGACCTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch     │ 0.8620346211761541 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AA

In [4]:

# Cell summary of all cells (counted 3x because run with 3 annotation tools)
cells.aggregate("cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("cell_count DESC")

┌────────────┬────────────────────────────────────┬────────────┐
│  cell_id   │             cell_label             │ cell_count │
│  varchar   │              varchar               │   int64    │
├────────────┼────────────────────────────────────┼────────────┤
│ CL:0002064 │ pancreatic acinar cell             │      16007 │
│ CL:0000173 │ pancreatic D cell                  │      13278 │
│ CL:0002079 │ pancreatic ductal cell             │       8413 │
│ CL:0000171 │ type B pancreatic cell             │       7385 │
│ CL:0002410 │ pancreatic stellate cell:quiescent │        732 │
│ CL:0000057 │ fibroblast                         │        655 │
│ CL:0000763 │ myeloid cell                       │        426 │
│ CL:0000738 │ leukocyte                          │        384 │
│ CL:0000115 │ endothelial cell                   │        324 │
│ CL:0002275 │ pancreatic PP cell                 │        281 │
│ CL:0000169 │ type B pancreatic cell             │        258 │
│ CL:0000084 │ T cell    

In [5]:
# Cell summaries by tool
summaries = cells.aggregate("tool, cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("tool, cell_count DESC")
summaries.to_df()

,tool,cell_id,cell_label,cell_count
0,azimuth,CL:0002064,pancreatic acinar cell,6721
1,azimuth,CL:0000171,type B pancreatic cell,4377
2,azimuth,CL:0002079,pancreatic ductal cell,3862
3,azimuth,CL:0002410,pancreatic stellate cell:quiescent,658
4,azimuth,CL:0000738,leukocyte,384
5,azimuth,CL:0000115,endothelial cell,151
6,azimuth,CL:0000173,pancreatic D cell,40
7,azimuth,CL:0002573,Schwann cell,18
8,azimuth,CL:0002275,pancreatic PP cell,11
9,celltypist,CL:0000173,pancreatic D cell,13238


In [ ]:
# Test reading the cell instances back in from csv
cells2 = duckdb.read_csv(f'{ data_dir }/cell-instances.csv.gz')
cells2.show()

┌────────────────────┬────────────────┬─────────┬──────────────────────────┬────────────┬────────────────────────┬─────────────────┬────────────────────┐
│      dataset       │     organ      │  tool   │           cell           │  cell_id   │       cell_label       │   match_type    │  confidence_score  │
│      varchar       │    varchar     │ varchar │         varchar          │  varchar   │        varchar         │     varchar     │       double       │
├────────────────────┼────────────────┼─────────┼──────────────────────────┼────────────┼────────────────────────┼─────────────────┼────────────────────┤
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGCACGTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch │ 0.7471968103744316 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGGACCTT-1-1JKYN │ CL:0002064 │ pancreatic acinar cell │ skos:exactMatch │ 0.8620346211761541 │
│ GTEX-PRIVATE-1JKYN │ UBERON:0001264 │ azimuth │ AAACAGCCAGGCGATA-1-1JKYN │